In [2]:
## GRU - D model to predict sepsis

import os
import time
import math
import random
import numpy as np
import pandas as pd
from typing import List, Optional

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, Sampler
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.metrics import precision_recall_curve, confusion_matrix
from sklearn.linear_model import LogisticRegression

# ---------------------------  hard-coded args  ------------------------
class Args: pass
args = Args()
args.train = "/content/drive/MyDrive/Sepsis Prediction/sepsis_dataset_padded_masked_train.csv"               # works with both raw and pre-padded/masked CSVs
args.val   = "/content/drive/MyDrive/Sepsis Prediction/sepsis_dataset_padded_masked_val.csv"
args.test  = "/content/drive/MyDrive/Sepsis Prediction/sepsis_dataset_padded_masked_test.csv"
args.pid_col = None
args.time_col = None
args.label_col = None

args.batch_size = 16
args.epochs = 100
args.lr = 1e-3
args.weight_decay = 1e-5
args.hidden_size = 128
args.layers = 2
args.dropout = 0.30
args.focal = True           # set False to use class-weighted BCE
args.seed = 42
args.save_model = "grud_model.pt"
args.grad_clip = 1.0
args.patience = 10          # early stop on patient-level AUPRC

# ---------------------------  seeds  ----------------------------------
torch.manual_seed(args.seed)
np.random.seed(args.seed)
random.seed(args.seed)

# ---------------------------  helpers --------------------------------
LABEL_CANDIDATES = ["sepsislabel", "sepsis_label", "label", "target", "y", "SepsisLabel"]
PID_CANDIDATES   = ["patientid", "pid", "subject_id", "stay_id", "id", "PatientID"]
TIME_CANDIDATES  = ["hour", "time", "t", "charttime", "timestamp", "offset", "Hour", "Step"]

META_COLS = {"PatientID", "Step", "Mask", "SepsisLabel"}

def detect_column(df: pd.DataFrame, candidates: List[str]) -> Optional[str]:
    cols_lower = {c.lower(): c for c in df.columns}
    for name in candidates:
        if name.lower() in cols_lower:
            return cols_lower[name.lower()]
    return None

def detect_pre_padded(df: pd.DataFrame) -> bool:
    return {"Step", "Mask"}.issubset(df.columns)

def select_feature_columns_from_pre_padded(df: pd.DataFrame):
    # numeric, not meta, not *_raw
    cols = []
    for c in df.columns:
        if c in META_COLS:
            continue
        if c.endswith("_raw"):   # discard raw columns
            continue
        if pd.api.types.is_numeric_dtype(df[c]):
            cols.append(c)
    return cols

def get_feature_columns(df, pid_col, time_col, label_col):
    exclude = {pid_col, time_col, label_col}
    return [c for c in df.columns if c not in exclude and pd.api.types.is_numeric_dtype(df[c])]

def to_elapsed_hours(series: pd.Series):
    if pd.api.types.is_numeric_dtype(series):
        return series.values.astype(np.float32)
    ts = pd.to_datetime(series, errors="coerce")
    if ts.isna().all():
        return np.arange(len(series), dtype=np.float32)
    base = ts.iloc[0]
    return ((ts - base).dt.total_seconds() / 3600.0).values.astype(np.float32)

def compute_deltas_per_feature(times, mask):
    """Per-feature time since last observation (mask 1 when observed)."""
    T, D = mask.shape
    delta = np.zeros((T, D), dtype=np.float32)
    last_time = np.full(D, times[0], dtype=np.float32)
    for t in range(T):
        delta[t] = times[t] - last_time
        last_time[mask[t] > 0.5] = times[t]
    return delta

def relabel_early_warning(df, pid_col, time_col, label_col, pre_hours=12):
    """
    Early-detection relabel: mark last `pre_hours` before onset as positive.
    Assumes time is monotone per patient.
    """
    def relabel_group(g):
        g = g.sort_values(time_col).copy()
        y = g[label_col].values
        pos_idx = np.where(y == 1)[0]
        if len(pos_idx) == 0:
            return g
        onset_i = pos_idx[0]
        onset_time = g[time_col].iloc[onset_i]
        window_min = onset_time - pre_hours
        g[label_col] = ((g[time_col] >= window_min) & (g[time_col] <= onset_time)).astype(np.float32)
        return g

    # pandas>=2.0 deprecates including grouping columns; here we keep default
    return df.groupby(pid_col, group_keys=False).apply(relabel_group).reset_index(drop=True)

# ---------------------------  dataset --------------------------------
class PatientSeriesDataset(Dataset):
    """
    Groups rows by patient, builds (X, M, Delta, y) sequences.
    - X: z-scored by TRAIN stats, NaNs -> 0 AFTER building mask M
    - M: mask 1 if originally observed, else 0 (after z-score); zero on padded rows
    - Delta: per-feature time since last observation (uses M)
    - y: binary labels (0/1) per timestep; padded/-1 -> 0 to satisfy BCE
    - step_mask: (optional) provided by pre-padded CSVs (1=real step, 0=padded)
    """
    def __init__(self, df, feature_cols, pid_col, time_col, label_col, feat_means, feat_stds):
        self._feat_means = np.asarray(feat_means, dtype=np.float32)
        self._feat_stds  = np.asarray(feat_stds,  dtype=np.float32)
        self.x_mean = torch.tensor(self._feat_means, dtype=torch.float32)  # GRU-D decays to mean

        has_mask = "Mask" in df.columns
        has_step = "Step" in df.columns

        self.groups = []
        for pid, g in df.groupby(pid_col):
            g = g.sort_values(time_col).copy()

            # times
            if has_step:
                times = g["Step"].values.astype(np.float32)
            else:
                times = to_elapsed_hours(g[time_col])

            # time-step mask and label
            if has_mask:
                step_mask = g["Mask"].values.astype(np.float32)
            else:
                step_mask = np.ones(len(g), dtype=np.float32)

            y = g[label_col].to_numpy(np.float32).reshape(-1, 1)
            # BCE expects valid targets everywhere; set padded or -1 labels to 0
            if has_mask:
                y = np.where(y < 0, 0.0, y).astype(np.float32).reshape(-1, 1)

            # features
            X = g[feature_cols].to_numpy(np.float32)

            # z-score (keep NaN to derive mask)
            X = (X - self._feat_means) / self._feat_stds

            # per-feature mask: observed & real time step
            M_feat = (~np.isnan(X)).astype(np.float32)
            M_feat = (M_feat.T * step_mask).T  # zero out padded rows

            # deltas per feature from mask and times
            Delta = compute_deltas_per_feature(times, M_feat)

            # fill NaNs in X with 0 (mask carries missingness)
            X = np.nan_to_num(X, nan=0.0)

            # Save group; true length will be derived from step_mask in collate
            self.groups.append(dict(
                pid=pid, X=X, M=M_feat, Delta=Delta, y=y, times=times, step_mask=step_mask
            ))

    def __len__(self): return len(self.groups)
    def __getitem__(self, i): return self.groups[i]

def collate_batch(batch):
    """Pad to max length in batch (works even if all are same T)."""
    B = len(batch)
    T_max = max(len(b["X"]) for b in batch)
    D = batch[0]["X"].shape[1]

    X = torch.zeros(B, T_max, D)
    M = torch.zeros(B, T_max, D)
    Delta = torch.zeros(B, T_max, D)
    y = torch.zeros(B, T_max, 1)
    lengths = torch.zeros(B, dtype=torch.long)

    for i, b in enumerate(batch):
        T = len(b["X"])
        X[i, :T] = torch.from_numpy(b["X"])
        M[i, :T] = torch.from_numpy(b["M"])
        Delta[i, :T] = torch.from_numpy(b["Delta"])
        y[i, :T] = torch.from_numpy(b["y"])
        # true (unpadded) length from time-step mask if present
        if "step_mask" in b:
            lengths[i] = int(b["step_mask"].sum())
        else:
            lengths[i] = T
    return X, M, Delta, y, lengths

class PatientBalancedSampler(Sampler):
    """
    Each batch draws ~half septic patients and ~half non-septic (by patient).
    """
    def __init__(self, dataset, batch_size=16, seed=42):
        self.batch_size = batch_size
        random.seed(seed)

        septic_idx, nonseptic_idx = [], []
        for i, g in enumerate(dataset.groups):
            y_seq = g["y"].reshape(-1)
            if (y_seq == 1).any():
                septic_idx.append(i)
            else:
                nonseptic_idx.append(i)
        self.septic_idx = septic_idx
        self.nonseptic_idx = nonseptic_idx

    def __iter__(self):
        k = max(1, self.batch_size // 2)
        s = self.septic_idx.copy(); n = self.nonseptic_idx.copy()
        random.shuffle(s); random.shuffle(n)
        si = ni = 0
        total = len(s) + len(n)
        emitted = 0
        while emitted < total:
            batch = []
            for _ in range(k):
                if si >= len(s): si = 0
                batch.append(s[si]); si += 1
            for _ in range(self.batch_size - k):
                if ni >= len(n): ni = 0
                batch.append(n[ni]); ni += 1
            random.shuffle(batch)
            for idx in batch:
                yield idx
            emitted += len(batch)

    def __len__(self):
        return len(self.septic_idx) + len(self.nonseptic_idx)

# ---------------------------  model -----------------------------------
class GRUDCell(nn.Module):
    def __init__(self, D, H):
        super().__init__()
        self.gamma_x_fc = nn.Linear(D, D)
        self.gamma_h_fc = nn.Linear(1, H)
        self.gru = nn.GRUCell(D*2, H)

    def forward(self, x_t, m_t, delta_t, h_prev, x_mean, x_prev_obs):
        # Feature decay toward population mean
        gamma_x = torch.exp(-torch.relu(self.gamma_x_fc(delta_t)))
        x_decay = gamma_x * x_prev_obs + (1 - gamma_x) * x_mean
        x_hat = m_t * x_t + (1 - m_t) * x_decay
        x_prev_obs_next = m_t * x_t + (1 - m_t) * x_prev_obs

        # Hidden decay by aggregated delta
        delta_agg = delta_t.mean(1, keepdim=True)
        gamma_h = torch.exp(-torch.relu(self.gamma_h_fc(delta_agg)))
        h_tilde = gamma_h * h_prev

        h_t = self.gru(torch.cat([x_hat, m_t], dim=1), h_tilde)
        return h_t, x_prev_obs_next

class GRUD(nn.Module):
    """
    Layer 0: GRU-D (uses masks/deltas/means)
    Higher layers: vanilla GRUCells on hidden states
    """
    def __init__(self, D, H, layers=1, dropout=0.1):
        super().__init__()
        assert layers >= 1
        self.D, self.H, self.layers = D, H, layers

        self.input_cell = GRUDCell(D, H)
        self.gru_layers = nn.ModuleList([nn.GRUCell(H, H) for _ in range(layers - 1)])

        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(H, 1)

    def forward(self, X, M, Delta, x_mean, lengths):
        B, T, D = X.shape
        device = X.device
        x_mean = x_mean.to(device).view(1, D)

        h0 = torch.zeros(B, self.H, device=device)
        x_prev_obs = torch.zeros(B, D, device=device)
        hs = [torch.zeros(B, self.H, device=device) for _ in range(len(self.gru_layers))]

        logits = torch.zeros(B, T, 1, device=device)

        for t in range(T):
            x_t = X[:, t, :]
            m_t = M[:, t, :]
            d_t = Delta[:, t, :]

            h0, x_prev_obs = self.input_cell(x_t, m_t, d_t, h0, x_mean, x_prev_obs)

            h = h0
            for i, cell in enumerate(self.gru_layers):
                h = cell(h, hs[i])
                hs[i] = h

            h_out = self.dropout(h)
            logits[:, t, 0] = self.fc(h_out).squeeze(1)

        # mask padded steps
        for i, L in enumerate(lengths):
            if L < T:
                logits[i, L:, 0] = -1e9
        return logits

# ---------------------------  losses & trains -------------------------
class FocalLoss(nn.Module):
    """
    Binary Focal Loss.
    alpha: weight for positive class (higher -> more recall)
    gamma: focusing parameter
    """
    def __init__(self, alpha=0.75, gamma=2.0, eps=1e-8):
        super().__init__()
        self.alpha = float(alpha)
        self.gamma = float(gamma)
        self.eps = float(eps)

    def forward(self, logits, y_true):
        p = torch.sigmoid(logits).clamp(self.eps, 1.0 - self.eps)
        ce = -(y_true * torch.log(p) + (1.0 - y_true) * torch.log(1.0 - p))
        p_t = y_true * p + (1.0 - y_true) * (1.0 - p)
        alpha_t = self.alpha * y_true + (1.0 - self.alpha) * (1.0 - y_true)
        loss = alpha_t * (1.0 - p_t).pow(self.gamma) * ce
        return loss.mean()

def compute_pos_weight(loader):
    pos = neg = 0
    for _, _, _, y, _ in loader:
        pos += (y == 1).sum().item()
        neg += (y == 0).sum().item()
    return max(1.0, neg / max(1.0, pos))

def run_epoch(model, loader, criterion, optimizer, x_mean, device, train=True):
    model.train(train)
    total = 0.0
    ys, ps = [], []
    for X, M, D, y, L in loader:
        X, M, D, y, L = X.to(device), M.to(device), D.to(device), y.to(device), L.to(device)
        logits = model(X, M, D, x_mean, L)
        loss = criterion(logits, y)
        if train:
            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), args.grad_clip)
            optimizer.step()
        total += loss.item() * X.size(0)
        with torch.no_grad():
            p = torch.sigmoid(logits).cpu().numpy().ravel()
            t = y.cpu().numpy().ravel()
            ys.append(t); ps.append(p)
    ys = np.concatenate(ys); ps = np.concatenate(ps)
    try: auroc = roc_auc_score(ys, ps)
    except: auroc = float('nan')
    try: aupr = average_precision_score(ys, ps)
    except: aupr = float('nan')
    return total / len(loader.dataset), auroc, aupr

# ---------------------------  patient-level metrics -------------------
def patient_level_scores(loader, model, x_mean, device, agg="max"):
    """
    Aggregate per-patient probabilities then compute AUROC/AUPRC.
    agg: "max" (best for early warning), "last", or "mean"
    """
    model.eval()
    ys, ps = [], []
    with torch.no_grad():
        for X, M, D, y, L in loader:
            X, M, D, y, L = X.to(device), M.to(device), D.to(device), y.to(device), L.to(device)
            prob = torch.sigmoid(model(X, M, D, x_mean, L)).squeeze(-1)  # (B,T)
            B = prob.shape[0]
            for i in range(B):
                T_i = int(L[i].item())
                p_seq = prob[i, :T_i]
                y_seq = y[i, :T_i, 0]
                y_patient = float((y_seq == 1).any().item())
                if agg == "last": p_patient = float(p_seq[-1].item())
                elif agg == "mean": p_patient = float(p_seq.mean().item())
                else: p_patient = float(p_seq.max().item())
                ys.append(y_patient); ps.append(p_patient)
    ys = np.array(ys); ps = np.array(ps)
    try:  auroc = roc_auc_score(ys, ps)
    except: auroc = float('nan')
    try:  auprc = average_precision_score(ys, ps)
    except:  auprc = float('nan')
    return auroc, auprc

def patient_level_probs(loader, model, x_mean, device, agg="max"):
    """Return arrays (y_true, y_score) aggregated per-patient."""
    model.eval()
    ys, ps = [], []
    with torch.no_grad():
        for X, M, D, y, L in loader:
            X, M, D, y, L = X.to(device), M.to(device), D.to(device), y.to(device), L.to(device)
            pr = torch.sigmoid(model(X, M, D, x_mean, L)).squeeze(-1)  # (B,T)
            B = pr.shape[0]
            for i in range(B):
                T_i = int(L[i].item())
                p_seq = pr[i, :T_i]; y_seq = y[i, :T_i, 0]
                y_patient = float((y_seq == 1).any().item())
                if agg == "last": p_patient = float(p_seq[-1].item())
                elif agg == "mean": p_patient = float(p_seq.mean().item())
                else: p_patient = float(p_seq.max().item())
                ys.append(y_patient); ps.append(p_patient)
    return np.array(ys), np.array(ps)

def metrics_at_threshold(y_true, y_score, thr):
    y_hat = (y_score >= thr).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_hat, labels=[0, 1]).ravel()
    prec = tp / max(1, (tp + fp))
    rec  = tp / max(1, (tp + fn))
    spec = tn / max(1, (tn + fp))
    return dict(threshold=thr, tp=tp, fp=fp, tn=tn, fn=fn,
                precision=prec, recall=rec, specificity=spec)

# ---------------------------  main -----------------------------------
def main():
    # Device & speed knobs
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    if device.type == "cuda":
        torch.backends.cudnn.benchmark = True  # autotune for current shapes

    # Load
    df_tr = pd.read_csv(args.train)
    df_va = pd.read_csv(args.val)
    df_te = pd.read_csv(args.test)

    pre_padded = detect_pre_padded(df_tr)

    # Detect columns (kept for backward-compatibility if not pre-padded)
    pid_col  = args.pid_col  or detect_column(df_tr, PID_CANDIDATES)   or "PatientID"
    time_col = args.time_col or detect_column(df_tr, TIME_CANDIDATES)  or ("Step" if pre_padded else "Hour")
    label_col  = args.label_col or detect_column(df_tr, LABEL_CANDIDATES) or "SepsisLabel"

    # If these CSVs are the padded/masked ones, DO NOT relabel
    if not pre_padded:
        df_tr = relabel_early_warning(df_tr, pid_col, time_col, label_col, pre_hours=12)
        df_va = relabel_early_warning(df_va, pid_col, time_col, label_col, pre_hours=12)
        df_te = relabel_early_warning(df_te, pid_col, time_col, label_col, pre_hours=12)

    # Feature columns and stats
    if pre_padded:
        feat_cols = select_feature_columns_from_pre_padded(df_tr)
        # compute stats on valid time steps only
        df_tr_stats = df_tr[df_tr["Mask"] == 1]
        feat_means_s = df_tr_stats[feat_cols].mean(skipna=True)
        feat_stds_s  = df_tr_stats[feat_cols].std(skipna=True).replace(0, 1e-6)
    else:
        feat_cols = get_feature_columns(df_tr, pid_col, time_col, label_col)
        feat_means_s = df_tr[feat_cols].mean(skipna=True)
        feat_stds_s  = df_tr[feat_cols].std(skipna=True).replace(0, 1e-6)

    feat_means = feat_means_s.values.astype(np.float32)
    feat_stds  = feat_stds_s.values.astype(np.float32)

    # Datasets
    tr = PatientSeriesDataset(df_tr, feat_cols, pid_col, time_col, label_col, feat_means, feat_stds)
    va = PatientSeriesDataset(df_va, feat_cols, pid_col, time_col, label_col, feat_means, feat_stds)
    te = PatientSeriesDataset(df_te, feat_cols, pid_col, time_col, label_col, feat_means, feat_stds)

    # Show dataset size (one-time)
    print(f"Patients: train={len(tr)} val={len(va)} test={len(te)} "
          f"| T={tr.groups[0]['X'].shape[0]} steps "
          f"| D={tr.groups[0]['X'].shape[1]} features")

    # DataLoader perf settings
    workers = max(2, (os.cpu_count() or 2) // 2)
    pin = (device.type == "cuda")
    train_loader = DataLoader(
        tr,
        batch_size=args.batch_size,
        sampler=PatientBalancedSampler(tr, batch_size=args.batch_size),
        collate_fn=collate_batch,
        num_workers=workers,
        pin_memory=pin,
        persistent_workers=True
    )
    val_loader   = DataLoader(
        va, batch_size=args.batch_size, shuffle=False, collate_fn=collate_batch,
        num_workers=workers, pin_memory=pin, persistent_workers=True
    )
    test_loader  = DataLoader(
        te, batch_size=args.batch_size, shuffle=False, collate_fn=collate_batch,
        num_workers=workers, pin_memory=pin, persistent_workers=True
    )

    # Model
    model = GRUD(len(feat_cols), args.hidden_size, args.layers, args.dropout).to(device)

    # Loss
    if args.focal:
        criterion = FocalLoss(alpha=0.65, gamma=1.0)
    else:
        posw = torch.tensor([compute_pos_weight(train_loader)], device=device)
        criterion = nn.BCEWithLogitsLoss(pos_weight=posw)

    # Optimizer
    opt = torch.optim.Adam(model.parameters(), lr=args.lr, weight_decay=args.weight_decay)

    # Train loop with patient-level model selection + timing + early stopping
    best_pl_aupr = -1.0
    best_state = None
    ema_epoch_time = None  # exponential moving average for ETA
    epochs_no_improve = 0

    for ep in range(1, args.epochs + 1):
        t0 = time.time()

        tl, _, ta = run_epoch(model, train_loader, criterion, opt, tr.x_mean, device, True)
        vl, _, va_ = run_epoch(model, val_loader,   criterion, opt, tr.x_mean, device, False)
        v_pl_auc, v_pl_aupr = patient_level_scores(val_loader, model, tr.x_mean, device, agg="max")

        elapsed = time.time() - t0
        ema_epoch_time = elapsed if ema_epoch_time is None else (0.3 * elapsed + 0.7 * ema_epoch_time)
        remaining = max(0.0, (args.epochs - ep) * (ema_epoch_time or elapsed))

        print(f"Epoch {ep:02d} | train loss {tl:.4f} AUPRC {ta:.4f} | "
              f"val loss {vl:.4f} AUPRC {va_:.4f} | "
              f"VAL (patient) AUROC {v_pl_auc:.4f} AUPRC {v_pl_aupr:.4f} | "
              f"time {elapsed:.1f}s | est left ~{remaining/60:.1f} min")

        if v_pl_aupr > best_pl_aupr:
            best_pl_aupr = v_pl_aupr
            best_state = {k: v.cpu() for k, v in model.state_dict().items()}
            epochs_no_improve = 0
        else:
            epochs_no_improve += 1
            if epochs_no_improve >= args.patience:
                print(f"Early stopping at epoch {ep} (no VAL patient AUPRC improvement for {args.patience} epochs).")
                break

    if best_state:
        model.load_state_dict(best_state)

    # ---- Calibration on VAL patient-level probs (Platt scaling) ----
    vy, vp = patient_level_probs(val_loader, model, tr.x_mean, device, agg="max")
    calib_lr = LogisticRegression(max_iter=500)
    calib_lr.fit(vp.reshape(-1, 1), vy.astype(int))

    def calibrated_probs(raw_probs: np.ndarray) -> np.ndarray:
        raw_probs = np.clip(raw_probs, 1e-8, 1 - 1e-8).reshape(-1, 1)
        return calib_lr.predict_proba(raw_probs)[:, 1]

    # ---- Threshold selection on CALIBRATED VAL probs ----
    vp_cal = calibrated_probs(vp)
    prec, rec, thr = precision_recall_curve(vy, vp_cal)
    f1 = 2 * prec * rec / np.clip(prec + rec, 1e-8, None)
    best_idx = int(np.nanargmax(f1))
    best_thr = float(thr[max(0, best_idx - 1)])
    print(f"Selected threshold on VAL (max F1, CAL): {best_thr:.4f} | "
          f"P={prec[best_idx]:.3f}, R={rec[best_idx]:.3f}, F1={np.nanmax(f1):.3f}")

    prec_a, rec_a, thr_a = prec[:-1], rec[:-1], thr  # align lengths
    mask = rec_a >= 0.90
    if mask.any():
        idx_c = int(np.argmax(prec_a[mask]))
        rc_thr = float(thr_a[mask][idx_c])
        rc_prec = float(prec_a[mask][idx_c]); rc_rec = float(rec_a[mask][idx_c])
    else:
        rc_thr = best_thr; rc_prec = float(prec[best_idx]); rc_rec = float(rec[best_idx])
    print(f"Selected VAL recall-constrained thr (rec>=0.90, CAL): {rc_thr:.4f} | "
          f"P={rc_prec:.3f}, R={rc_rec:.3f}")

    # ---- Event-level TEST (per-timestep) ----
    l, au, ap = run_epoch(model, test_loader, criterion, opt, tr.x_mean, device, False)
    print(f"\nTEST: loss={l:.4f} AUROC={au:.4f} AUPRC={ap:.4f}")

    # ---- Patient-level TEST AUCs ----
    t_pl_auc, t_pl_aupr = patient_level_scores(test_loader, model, tr.x_mean, device, agg="max")
    print(f"PATIENT-level TEST: AUROC={t_pl_auc:.4f} AUPRC={t_pl_aupr:.4f}")

    # Save best model weights
    torch.save(model.state_dict(), args.save_model)
    print("Saved model to", args.save_model)

    # ---- Thresholded TEST metrics (CALIBRATED) ----
    ty, tp = patient_level_probs(test_loader, model, tr.x_mean, device, agg="max")
    tp_cal = calibrated_probs(tp)

    m = metrics_at_threshold(ty, tp_cal, best_thr)
    print(f"PATIENT-level TEST @thr={m['threshold']:.4f} | "
          f"Precision={m['precision']:.3f} Recall={m['recall']:.3f} "
          f"Specificity={m['specificity']:.3f} | "
          f"TP={m['tp']} FP={m['fp']} TN={m['tn']} FN={m['fn']}")

    m_rc = metrics_at_threshold(ty, tp_cal, rc_thr)
    print(f"PATIENT-level TEST @rec>=0.90 thr={m_rc['threshold']:.4f} | "
          f"Precision={m_rc['precision']:.3f} Recall={m_rc['recall']:.3f} "
          f"Specificity={m_rc['specificity']:.3f} | "
          f"TP={m_rc['tp']} FP={m_rc['fp']} TN={m_rc['tn']} FN={m_rc['fn']}")

if __name__ == "__main__":
    main()


Patients: train=24201 val=8068 test=8067 | T=58 steps | D=40 features
Epoch 01 | train loss 0.0227 AUPRC 0.5362 | val loss 0.0153 AUPRC 0.3446 | VAL (patient) AUROC 0.8207 AUPRC 0.4353 | time 238.3s | est left ~393.3 min
Epoch 02 | train loss 0.0193 AUPRC 0.6434 | val loss 0.0145 AUPRC 0.3645 | VAL (patient) AUROC 0.8332 AUPRC 0.4700 | time 237.5s | est left ~388.9 min
Epoch 03 | train loss 0.0184 AUPRC 0.6721 | val loss 0.0120 AUPRC 0.3864 | VAL (patient) AUROC 0.8321 AUPRC 0.4869 | time 238.3s | est left ~385.0 min
Epoch 04 | train loss 0.0180 AUPRC 0.6840 | val loss 0.0106 AUPRC 0.3716 | VAL (patient) AUROC 0.7971 AUPRC 0.4598 | time 237.7s | est left ~380.8 min
Epoch 05 | train loss 0.0175 AUPRC 0.6951 | val loss 0.0159 AUPRC 0.3694 | VAL (patient) AUROC 0.8220 AUPRC 0.4686 | time 236.8s | est left ~376.3 min
Epoch 06 | train loss 0.0173 AUPRC 0.7010 | val loss 0.0118 AUPRC 0.3510 | VAL (patient) AUROC 0.8308 AUPRC 0.4901 | time 237.1s | est left ~372.1 min
Epoch 07 | train loss 0.